In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from pathlib import Path
import sys

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

SRC_PATH = PROJECT_PATH / "src"

DATASET_PATH = PROJECT_PATH / "datasets" / "raw" / "UAH-DRIVESET-v1"

sys.path.append(str(SRC_PATH))

In [ ]:
import importlib

import data_loader
import lstm_model

importlib.reload(data_loader)
importlib.reload(lstm_model)

<module 'lstm_model' from '/content/drive/MyDrive/UAH_Project/src/lstm_model.py'>

In [ ]:
X, y, groups = data_loader.build_dataset(DATASET_PATH)

print(X.shape)
print(y.shape)
print(groups.shape)

Dataset Shape : (30676, 120, 13)
Labels        : (30676,)
Groups        : (30676,)
(30676, 120, 13)
(30676,)
(30676,)


In [ ]:
import os
import numpy as np

save_dir = PROJECT_PATH / "datasets" / "processed"
save_dir.mkdir(parents=True, exist_ok=True)

save_path = save_dir / "uah_dataset.npz"

np.savez(
    save_path,
    X=X,
    y=y,
    groups=groups,
)

print("Dataset saved to:")
print(save_path)

Dataset saved to:
/content/drive/MyDrive/UAH_Project/datasets/processed/uah_dataset.npz


In [ ]:
data = np.load(save_path)

print(data.files)

print(data["X"].shape)
print(data["y"].shape)
print(data["groups"].shape)

['X', 'y', 'groups']
(30676, 120, 13)
(30676,)
(30676,)


In [ ]:
import importlib
import trainer

importlib.reload(trainer)

<module 'trainer' from '/content/drive/MyDrive/UAH_Project/src/trainer.py'>

In [ ]:
from pathlib import Path

trainer_path = PROJECT_PATH / "src" / "trainer.py"

print(trainer_path.read_text())

"""
trainer.py

Training utilities for UAH Driver Risk Prediction.
"""

import torch
import torch.nn as nn

from torch.utils.data import Dataset

class UAHDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]


In [ ]:
import trainer

print(dir(trainer))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [ ]:
from trainer import UAHDataset

dataset = UAHDataset(X, y)

print(len(dataset))

sample_x, sample_y = dataset[0]

print(sample_x.shape)
print(sample_y)

30676
torch.Size([120, 13])
tensor(0)


In [ ]:
import importlib
import trainer

importlib.reload(trainer)

<module 'trainer' from '/content/drive/MyDrive/UAH_Project/src/trainer.py'>

In [ ]:
from trainer import create_dataloader

train_loader = create_dataloader(
    X,
    y,
    batch_size=32,
)

print(len(train_loader))

959


In [ ]:
batch_x, batch_y = next(iter(train_loader))

print(batch_x.shape)
print(batch_y.shape)

print(batch_y[:10])

torch.Size([32, 120, 13])
torch.Size([32])
tensor([0, 1, 0, 0, 2, 1, 1, 0, 0, 2])


In [ ]:
from lstm_model import LSTMClassifier

model = LSTMClassifier()

outputs = model(batch_x)

print(outputs.shape)

print(outputs[:5])

torch.Size([32, 3])
tensor([[-0.0186,  0.0975, -0.2516],
        [ 0.0191, -0.0193, -0.1042],
        [ 0.0292,  0.0516, -0.1922],
        [ 0.0287,  0.0336, -0.1883],
        [ 0.0120,  0.0867, -0.2267]], grad_fn=<SliceBackward0>)


In [ ]:
criterion = nn.CrossEntropyLoss()

loss = criterion(outputs, batch_y)

print(loss)

tensor(1.0951, grad_fn=<NllLossBackward0>)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMClassifier().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

In [ ]:
from trainer import train_one_epoch

In [ ]:
loss, acc = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
)

print(loss)
print(acc)

0.9366919451759307
0.5169187638544791


In [ ]:
import importlib
import trainer

importlib.reload(trainer)

<module 'trainer' from '/content/drive/MyDrive/UAH_Project/src/trainer.py'>

In [ ]:
from trainer import validate_one_epoch

In [ ]:
val_loss, val_acc = validate_one_epoch(
    model,
    train_loader,
    criterion,
    device,
)

print(val_loss)
print(val_acc)

0.8816378607615689
0.541661233537619


In [ ]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import fit

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=3,
)

Epoch 1/3 | Train Loss: 0.8340 | Train Acc: 0.5803 | Val Loss: 0.7794 | Val Acc: 0.6098
Epoch 2/3 | Train Loss: 0.7981 | Train Acc: 0.6063 | Val Loss: 0.7055 | Val Acc: 0.6591
Epoch 3/3 | Train Loss: 0.7210 | Train Acc: 0.6463 | Val Loss: 0.6632 | Val Acc: 0.6601


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [ ]:
gkf = GroupKFold(n_splits=5)

print(gkf)

GroupKFold(n_splits=5, random_state=None, shuffle=False)


In [ ]:
for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X, y, groups)
):

    print(f"Fold {fold + 1}")

    print("Train:", len(train_idx))
    print("Validation:", len(val_idx))

    break

Fold 1
Train: 24572
Validation: 6104


In [ ]:
train_groups = set(groups[train_idx])
val_groups = set(groups[val_idx])

intersection = train_groups.intersection(val_groups)

print("Train Trip:", len(train_groups))
print("Validation Trip:", len(val_groups))
print("Common Trips:", len(intersection))

Train Trip: 32
Validation Trip: 8
Common Trips: 0


In [ ]:
X_train = X[train_idx]
y_train = y[train_idx]

X_val = X[val_idx]
y_val = y[val_idx]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

Train: (24572, 120, 13)
Validation: (6104, 120, 13)


In [ ]:
from trainer import create_dataloader

train_loader = create_dataloader(
    X_train,
    y_train,
    batch_size=32,
    shuffle=True,
)

val_loader = create_dataloader(
    X_val,
    y_val,
    batch_size=32,
    shuffle=False,
)

print(len(train_loader))
print(len(val_loader))

768
191


In [ ]:
from lstm_model import LSTMClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMClassifier().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=5,
)

Epoch 1/5 | Train Loss: 0.9013 | Train Acc: 0.5272 | Val Loss: 1.2089 | Val Acc: 0.3747
Epoch 2/5 | Train Loss: 0.7551 | Train Acc: 0.6234 | Val Loss: 0.9740 | Val Acc: 0.5541
Epoch 3/5 | Train Loss: 0.7121 | Train Acc: 0.6578 | Val Loss: 1.1782 | Val Acc: 0.3799
Epoch 4/5 | Train Loss: 0.6641 | Train Acc: 0.6974 | Val Loss: 1.0597 | Val Acc: 0.5070
Epoch 5/5 | Train Loss: 0.6264 | Train Acc: 0.7180 | Val Loss: 1.2286 | Val Acc: 0.4669


In [ ]:
gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X, y, groups)
):

    print("=" * 60)
    print(f"Fold {fold + 1}")
    print("=" * 60)

    print(len(train_idx))
    print(len(val_idx))

    print()

Fold 1
24572
6104

Fold 2
24480
6196

Fold 3
24633
6043

Fold 4
24544
6132

Fold 5
24475
6201



In [ ]:
gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X, y, groups)
):

    print("=" * 60)
    print(f"Fold {fold + 1}")
    print("=" * 60)

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_val = X[val_idx]
    y_val = y[val_idx]

    print("Train :", X_train.shape)
    print("Validation :", X_val.shape)

    print()

Fold 1
Train : (24572, 120, 13)
Validation : (6104, 120, 13)

Fold 2
Train : (24480, 120, 13)
Validation : (6196, 120, 13)

Fold 3
Train : (24633, 120, 13)
Validation : (6043, 120, 13)

Fold 4
Train : (24544, 120, 13)
Validation : (6132, 120, 13)

Fold 5
Train : (24475, 120, 13)
Validation : (6201, 120, 13)



In [ ]:
gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X, y, groups)
):

    print("=" * 60)
    print(f"Fold {fold + 1}")
    print("=" * 60)

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_val = X[val_idx]
    y_val = y[val_idx]

    print("Train :", X_train.shape)
    print("Validation :", X_val.shape)

    train_loader = create_dataloader(
        X_train,
        y_train,
        batch_size=32,
        shuffle=True,
    )

    val_loader = create_dataloader(
        X_val,
        y_val,
        batch_size=32,
        shuffle=False,
    )

    print("Train Loader :", len(train_loader))
    print("Val Loader   :", len(val_loader))

    print()

    print()

Fold 1
Train : (24572, 120, 13)
Validation : (6104, 120, 13)
Train Loader : 768
Val Loader   : 191


Fold 2
Train : (24480, 120, 13)
Validation : (6196, 120, 13)
Train Loader : 765
Val Loader   : 194


Fold 3
Train : (24633, 120, 13)
Validation : (6043, 120, 13)
Train Loader : 770
Val Loader   : 189


Fold 4
Train : (24544, 120, 13)
Validation : (6132, 120, 13)
Train Loader : 767
Val Loader   : 192


Fold 5
Train : (24475, 120, 13)
Validation : (6201, 120, 13)
Train Loader : 765
Val Loader   : 194




In [ ]:
fold_histories = []

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(X, y, groups)
):

    print("=" * 70)
    print(f"Fold {fold + 1}")
    print("=" * 70)

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_val = X[val_idx]
    y_val = y[val_idx]

    train_loader = create_dataloader(
        X_train,
        y_train,
        batch_size=32,
        shuffle=True,
    )

    val_loader = create_dataloader(
        X_val,
        y_val,
        batch_size=32,
        shuffle=False,
    )

    model = LSTMClassifier().to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
    )

    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=5,
    )

    fold_histories.append(history)

Fold 1
Epoch 1/5 | Train Loss: 0.8911 | Train Acc: 0.5395 | Val Loss: 1.0889 | Val Acc: 0.3684
Epoch 2/5 | Train Loss: 0.7796 | Train Acc: 0.5980 | Val Loss: 1.5437 | Val Acc: 0.4223
Epoch 3/5 | Train Loss: 0.7411 | Train Acc: 0.6240 | Val Loss: 1.3517 | Val Acc: 0.4633
Epoch 4/5 | Train Loss: 0.7629 | Train Acc: 0.6081 | Val Loss: 1.3041 | Val Acc: 0.4135
Epoch 5/5 | Train Loss: 0.7044 | Train Acc: 0.6478 | Val Loss: 1.1581 | Val Acc: 0.4663
Fold 2
Epoch 1/5 | Train Loss: 0.8957 | Train Acc: 0.5403 | Val Loss: 1.0130 | Val Acc: 0.4992
Epoch 2/5 | Train Loss: 0.7895 | Train Acc: 0.6134 | Val Loss: 0.8606 | Val Acc: 0.5205
Epoch 3/5 | Train Loss: 0.7038 | Train Acc: 0.6586 | Val Loss: 0.9700 | Val Acc: 0.4487
Epoch 4/5 | Train Loss: 0.6642 | Train Acc: 0.6802 | Val Loss: 0.9676 | Val Acc: 0.4797
Epoch 5/5 | Train Loss: 0.6346 | Train Acc: 0.6988 | Val Loss: 0.9551 | Val Acc: 0.4958
Fold 3
Epoch 1/5 | Train Loss: 0.8571 | Train Acc: 0.5823 | Val Loss: 1.3167 | Val Acc: 0.3093
Epoch 2/5 |

In [ ]:
best_val_accs = []

for history in fold_histories:
    best_val_accs.append(max(history["val_acc"]))

print(best_val_accs)

print()

print("Average Validation Accuracy:",
      sum(best_val_accs) / len(best_val_accs))

[0.4662516382699869, 0.5204970948999355, 0.41452920734734405, 0.4481409001956947, 0.5616835994194485]

Average Validation Accuracy: 0.48222048802648193


In [ ]:
from pathlib import Path

trainer_path = PROJECT_PATH / "src" / "trainer.py"

print(trainer_path.read_text()[-1200:])

(predictions.cpu().numpy())
            all_labels.extend(batch_y.numpy())

    return all_labels, all_predictions

def fit(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs,
):
    """
    Train the model for multiple epochs.
    """

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(epochs):

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
        )

        val_loss, val_acc = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device,
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch+1}/{epochs} | "
  

In [ ]:
import importlib
import trainer

importlib.reload(trainer)

print(dir(trainer))

['DataLoader', 'Dataset', 'UAHDataset', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'accuracy_score', 'create_dataloader', 'evaluate_model', 'f1_score', 'fit', 'nn', 'precision_score', 'recall_score', 'torch', 'train_one_epoch', 'validate_one_epoch']


In [ ]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import evaluate_model

In [ ]:
labels, predictions = evaluate_model(
    model,
    val_loader,
    device,
)

print(len(labels))
print(len(predictions))

6201
6201


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        labels,
        predictions,
        target_names=[
            "NORMAL",
            "AGGRESSIVE",
            "DROWSY",
        ]
    )
)

              precision    recall  f1-score   support

      NORMAL       0.52      0.56      0.54      2557
  AGGRESSIVE       0.62      0.49      0.55      2389
      DROWSY       0.56      0.70      0.63      1255

    accuracy                           0.56      6201
   macro avg       0.57      0.58      0.57      6201
weighted avg       0.57      0.56      0.56      6201



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, predictions)

print(cm)

[[1435  592  530]
 [1069 1164  156]
 [ 248  123  884]]


In [ ]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import fit

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=5,
)

Epoch 1/5 | Train Loss: 0.6753 | Train Acc: 0.6905 | Val Loss: 0.7566 | Val Acc: 0.6091
Epoch 2/5 | Train Loss: 0.6432 | Train Acc: 0.7118 | Val Loss: 1.0306 | Val Acc: 0.5081
Epoch 3/5 | Train Loss: 0.6222 | Train Acc: 0.7272 | Val Loss: 0.8153 | Val Acc: 0.6207
Epoch 4/5 | Train Loss: 0.5965 | Train Acc: 0.7443 | Val Loss: 0.8397 | Val Acc: 0.5854
Epoch 5/5 | Train Loss: 0.5475 | Train Acc: 0.7638 | Val Loss: 1.0031 | Val Acc: 0.5788


In [ ]:
print(history["best_val_acc"])

0.6207063376874697


In [ ]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import fit

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=20,
    patience=3,
)

Epoch 1/20 | Train Loss: 0.5531 | Train Acc: 0.7626 | Val Loss: 0.9258 | Val Acc: 0.5826
Epoch 2/20 | Train Loss: 0.5157 | Train Acc: 0.7830 | Val Loss: 0.9070 | Val Acc: 0.6230
Epoch 3/20 | Train Loss: 0.4721 | Train Acc: 0.8004 | Val Loss: 1.1657 | Val Acc: 0.5636
Epoch 4/20 | Train Loss: 0.4401 | Train Acc: 0.8154 | Val Loss: 1.0392 | Val Acc: 0.5896
Epoch 5/20 | Train Loss: 0.4172 | Train Acc: 0.8228 | Val Loss: 1.1879 | Val Acc: 0.5572

Early stopping at epoch 5
Best Validation Accuracy : 0.6230
